In [ ]:
import os, re, difflib
import numpy as np
import pandas as pd
from glob import glob
from statsmodels.tsa.filters.hp_filter import hpfilter
from scipy.signal import find_peaks

# 1. 参数设置
base = r"D:\研\大论文\数据\热度数据"
influence_path = os.path.join(base, "影响因素分析_213起事件综合表.xlsx")
out_path = os.path.join(base, "最终整合结果", "七项演化特征指标_213事件终版.csv")
folders = [os.path.join(base, "分年份数据", f"单事件小时数据{y}") for y in range(2021, 2027)]
LAMBDA = 100

# 2. 数据读取
def clean_name(s):
    s = str(s).lower()
    for x in ["微博源数据", "微信网媒源数据", "_小时数据", "小时数据"]:
        s = s.replace(x, "")
    return re.sub(r"[^\u4e00-\u9fa50-9]", "", s)

def read_csv(file):
    for enc in ["utf-8-sig", "gb18030", "gbk", "utf-8"]:
        try:
            return pd.read_csv(file, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"无法识别编码：{file}")

# 3. 正式事件名单
df_infl = pd.read_excel(influence_path)
col = next((c for c in df_infl.columns if "事件" in str(c) or "名称" in str(c)), None)
if col is None:
    raise ValueError("未找到事件名称列")

valid_names = df_infl[col].dropna().astype(str).drop_duplicates().tolist()
valid_pairs = [(clean_name(x), x) for x in valid_names if clean_name(x)]

# 4. HP滤波及七项指标提取
results, skipped = [], []

for folder in folders:
    year = os.path.basename(folder)[-4:]
    for file in glob(os.path.join(folder, "*.csv")):
        name = os.path.basename(file).replace(".csv", "")

        try:
            df = read_csv(file)
            if "发文量" not in df.columns:
                raise ValueError("缺少发文量列")

            y = pd.to_numeric(df["发文量"], errors="coerce").dropna().to_numpy(float)
            if len(y) < 10 or not np.isfinite(y).all():
                raise ValueError("有效数据不足或存在非有限值")

            cycle, trend = hpfilter(y, lamb=LAMBDA)
            n = len(trend)
            max_idx = np.argmax(trend)
            max_trend = np.max(trend)

            peaks, _ = find_peaks(trend, height=max_trend * 0.5)
            decay_idx = next(
                (i for i in range(max_idx + 1, n) if trend[i] < max_trend / 2),
                max_idx
            )

            results.append({
                "事件": name,
                "年份": year,
                "峰值数": len(peaks),
                "首次峰值时间": peaks[0] / n if len(peaks) else 0,
                "最大峰值时间": max_idx / n,
                "最大振幅": max_trend,
                "衰减时间": (decay_idx - max_idx) / n,
                "波动强度": np.std(cycle),
                "持续时长": n
            })

        except Exception as e:
            skipped.append((name, str(e)))

df_feat = pd.DataFrame(results)

# 5. 正式事件名称匹配
def match_event(name):
    c = clean_name(name)
    if not c:
        return None, 0, "未匹配", False

    candidates = []
    for key, original in valid_pairs:
        if c == key:
            score, method = 1.0, "完全匹配"
        elif c in key or key in c:
            score = min(len(c), len(key)) / max(len(c), len(key))
            method = "包含匹配"
        else:
            score = difflib.SequenceMatcher(None, c, key).ratio()
            method = "相似度匹配"

        if score >= 0.8:
            candidates.append((score, original, method))

    if not candidates:
        return None, 0, "未匹配", False

    candidates.sort(key=lambda x: x[0], reverse=True)
    best = candidates[0]
    ambiguous = any(
        abs(x[0] - best[0]) < 1e-10 and x[1] != best[1]
        for x in candidates[1:]
    )
    return best[1], best[0], best[2], ambiguous

match_info = df_feat["事件"].apply(match_event)
df_feat[["匹配事件", "匹配相似度", "匹配方式", "匹配歧义"]] = pd.DataFrame(
    match_info.tolist(), index=df_feat.index
)

# 6. 筛选、去重及排序
df_matched = df_feat[
    df_feat["匹配事件"].notna() & ~df_feat["匹配歧义"]
].sort_values("匹配相似度", ascending=False).copy()

duplicate = df_matched[
    df_matched.duplicated("匹配事件", keep=False)
].copy()

df_final = df_matched.drop_duplicates("匹配事件", keep="first").copy()
order = {x: i for i, x in enumerate(valid_names)}
df_final["_order"] = df_final["匹配事件"].map(order)
df_final = df_final.sort_values("_order").drop(columns="_order")
df_final = df_final.rename(columns={"匹配事件": "标准事件名称"})

missing = [x for x in valid_names if x not in set(df_final["标准事件名称"])]
audit = df_feat[["事件", "匹配事件", "匹配相似度", "匹配方式", "匹配歧义"]]

# 7. 保存结果
os.makedirs(os.path.dirname(out_path), exist_ok=True)

df_final.to_csv(out_path, index=False, encoding="utf-8-sig")
audit.to_csv(out_path.replace(".csv", "_匹配核查.csv"), index=False, encoding="utf-8-sig")
pd.DataFrame({"未匹配事件": missing}).to_csv(
    out_path.replace(".csv", "_未匹配事件.csv"), index=False, encoding="utf-8-sig"
)
duplicate.to_csv(out_path.replace(".csv", "_重复映射.csv"), index=False, encoding="utf-8-sig")
pd.DataFrame(skipped, columns=["事件", "失败原因"]).to_csv(
    out_path.replace(".csv", "_提取失败.csv"), index=False, encoding="utf-8-sig"
)

# 8. 结果汇总
print(f"正式名单：{len(valid_names)}")
print(f"成功提取：{len(df_feat)}")
print(f"最终唯一事件：{len(df_final)}")
print(f"未匹配：{len(missing)}")
print(f"重复映射：{len(duplicate)}")
print(f"提取失败：{len(skipped)}")

if len(df_final) == 213 and not missing and duplicate.empty:
    print("成功：已得到213起唯一事件的七项演化特征。")
else:
    print("警告：结果尚未达到213起，请检查核查文件。")

print(f"保存路径：{out_path}")